# Transforming

In this notebook, we will explore how to use Large Language Models for text transformation tasks such as language translation, spelling and grammar checking, tone adjustment, and format conversion.

## Setup

In [1]:
import openai
import os

from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv()) # read local .env file

openai.api_key  = os.getenv('OPENAI_API_KEY')

In [2]:
def get_completion(prompt, model="gpt-3.5-turbo", temperature=0): 
    messages = [{"role": "user", "content": prompt}]
    response = openai.ChatCompletion.create(
        model=model,
        messages=messages,
        temperature=temperature, 
    )
    return response.choices[0].message["content"]

## Translation

ChatGPT is trained with sources in many languages. This gives the model the ability to do translation. Here are some examples of how to use this capability.

In [3]:
prompt = f"""
Translate the following English text to Spanish: \ 
```Hi, I would like to order a blender```
"""
response = get_completion(prompt)
print(response)

Hola, me gustaría ordenar una licuadora.


In [4]:
prompt = f"""
Tell me which language this is: 
```Combien coûte le lampadaire?```
"""
response = get_completion(prompt)
print(response)

This is French.


In [5]:
prompt = f"""
Translate the following  text to French and Spanish
and English pirate: \
```I want to order a basketball```
"""
response = get_completion(prompt)
print(response)

French: Je veux commander un ballon de basket
Spanish: Quiero ordenar un balón de baloncesto
English: I want to order a basketball


In [6]:
prompt = f"""
Translate the following text to Spanish in both the \
formal and informal forms: 
'Would you like to order a pillow?'
"""
response = get_completion(prompt)
print(response)

Formal: ¿Le gustaría ordenar una almohada?
Informal: ¿Te gustaría ordenar una almohada?


### Universal Translator
Imagine you are in charge of IT at a large multinational e-commerce company. Users are messaging you with IT issues in all their native languages. Your staff is from all over the world and speaks only their native languages. You need a universal translator!

In [7]:
user_messages = [
  "La performance du système est plus lente que d'habitude.",  # System performance is slower than normal         
  "Mi monitor tiene píxeles que no se iluminan.",              # My monitor has pixels that are not lighting
  "Il mio mouse non funziona",                                 # My mouse is not working
  "Mój klawisz Ctrl jest zepsuty",                             # My keyboard has a broken control key
  "我的屏幕在闪烁"                                               # My screen is flashing
] 

In [8]:
for issue in user_messages:
    prompt = f"Tell me what language this is: ```{issue}```"
    lang = get_completion(prompt)
    print(f"Original message ({lang}): {issue}")

    prompt = f"""
    Translate the following  text to English \
    and Korean: ```{issue}```
    """
    response = get_completion(prompt)
    print(response, "\n")

Original message (This is French.): La performance du système est plus lente que d'habitude.
English: "The system performance is slower than usual."

Korean: "시스템 성능이 평소보다 느립니다." 

Original message (This is Spanish.): Mi monitor tiene píxeles que no se iluminan.
English: "My monitor has pixels that do not light up."

Korean: "내 모니터에는 빛나지 않는 픽셀이 있습니다." 

Original message (Italian): Il mio mouse non funziona
English: My mouse is not working
Korean: 내 마우스가 작동하지 않습니다 

Original message (Polish): Mój klawisz Ctrl jest zepsuty
English: My Ctrl key is broken
Korean: 제 Ctrl 키가 고장 났어요 

Original message (This is Chinese.): 我的屏幕在闪烁
English: My screen is flickering
Korean: 내 화면이 깜박거립니다 



## Try it yourself!
Try some translations on your own!

## Tone Transfermation

In [9]:
customer_message = """
Hey, I need help with my order. The package was supposed to arrive
yesterday, but I still haven't received it. Can you please check
where it is and let me know when I can expect it?
"""

prompt = f"""
Transform the following informal customer message into a
professional and polite business email.

Requirements:
- Keep the original meaning.
- Use professional and polite language.
- Do not add any information that is not in the original message.
- Keep it concise.

Customer message:
```{customer_message}```
"""

response = get_completion(prompt)

print(response)

Dear [Customer's Name],

I hope this email finds you well. I am reaching out regarding your recent order. According to the tracking information, the package was scheduled to be delivered yesterday. I apologize for any inconvenience this delay may have caused.

I will investigate the current status of your order and provide you with an update on when you can expect to receive it. Thank you for bringing this to our attention.

Best regards,

[Your Name]
[Your Position]
[Your Contact Information]


## Tone Transformation
Writing can vary based on the intended audience. ChatGPT can produce different tones.


In [ ]:
prompt = f"""
Translate the following from slang to a business letter: 
'Dude, This is Joe, check out this spec on this standing lamp.'
"""
response = get_completion(prompt)
print(response)

## Observation:
#### The experiment demonstrates that an LLM can transform informal text into professional business language while preserving the original meaning. By explicitly specifying the desired tone, audience, and constraints, the model produced a polite and structured business email. This shows that prompt instructions can control the tone and style of generated text without changing its core message.



### Meaning       → Same
### Information   → Mostly same
### Tone          → Changed
### Style         → Changed
### Format        → Changed

## Format Conversion
ChatGPT can translate between formats. The prompt should describe the input and output formats.

In [ ]:
data_json = { "resturant employees" :[ 
    {"name":"Shyam", "email":"shyamjaiswal@gmail.com"},
    {"name":"Bob", "email":"bob32@gmail.com"},
    {"name":"Jai", "email":"jai87@gmail.com"}
]}

prompt = f"""
Translate the following python dictionary from JSON to an HTML \
table with column headers and title: {data_json}
"""
response = get_completion(prompt)
print(response)

In [ ]:
from IPython.display import display, Markdown, Latex, HTML, JSON
display(HTML(response))

## Spellcheck/Grammar check.

Here are some examples of common grammar and spelling problems and the LLM's response. 

To signal to the LLM that you want it to proofread your text, you instruct the model to 'proofread' or 'proofread and correct'.

In [ ]:
text = [ 
  "The girl with the black and white puppies have a ball.",  # The girl has a ball.
  "Yolanda has her notebook.", # ok
  "Its going to be a long day. Does the car need it’s oil changed?",  # Homonyms
  "Their goes my freedom. There going to bring they’re suitcases.",  # Homonyms
  "Your going to need you’re notebook.",  # Homonyms
  "That medicine effects my ability to sleep. Have you heard of the butterfly affect?", # Homonyms
  "This phrase is to cherck chatGPT for speling abilitty"  # spelling
]
for t in text:
    prompt = f"""Proofread and correct the following text
    and rewrite the corrected version. If you don't find
    and errors, just say "No errors found". Don't use 
    any punctuation around the text:
    ```{t}```"""
    response = get_completion(prompt)
    print(response)

In [ ]:
text = f"""
Got this for my daughter for her birthday cuz she keeps taking \
mine from my room.  Yes, adults also like pandas too.  She takes \
it everywhere with her, and it's super soft and cute.  One of the \
ears is a bit lower than the other, and I don't think that was \
designed to be asymmetrical. It's a bit small for what I paid for it \
though. I think there might be other options that are bigger for \
the same price.  It arrived a day earlier than expected, so I got \
to play with it myself before I gave it to my daughter.
"""
prompt = f"proofread and correct this review: ```{text}```"
response = get_completion(prompt)
print(response)

In [ ]:
from redlines import Redlines

diff = Redlines(text,response)
display(Markdown(diff.output_markdown))

In [ ]:
prompt = f"""
proofread and correct this review. Make it more compelling. 
Ensure it follows APA style guide and targets an advanced reader. 
Output in markdown format.
Text: ```{text}```
"""
response = get_completion(prompt)
display(Markdown(response))

## Try it yourself!
Try changing the instructions to form your own review.

### Formate Transfermation

In [10]:
employees = {
    "employees": [
        {"name": "Ali", "department": "AI", "experience": 2},
        {"name": "Sara", "department": "Data Science", "experience": 3},
        {"name": "Hamza", "department": "Software Engineering", "experience": 1}
    ]
}

prompt = f"""
Convert the following employee data into an HTML table.

Requirements:
- Include a title: "Employee Information"
- Use these column headers:
  Name, Department, Experience
- Create one row for each employee.
- Return only the HTML code.
- Do not add or change any information.

Employee data:
```{employees}```
"""

response = get_completion(prompt)

print(response)

<!DOCTYPE html>
<html>
<head>
    <title>Employee Information</title>
</head>
<body>
    <table>
        <tr>
            <th>Name</th>
            <th>Department</th>
            <th>Experience</th>
        </tr>
        <tr>
            <td>Ali</td>
            <td>AI</td>
            <td>2</td>
        </tr>
        <tr>
            <td>Sara</td>
            <td>Data Science</td>
            <td>3</td>
        </tr>
        <tr>
            <td>Hamza</td>
            <td>Software Engineering</td>
            <td>1</td>
        </tr>
    </table>
</body>
</html>


Thanks to the following sites:

https://writingprompts.com/bad-grammar-examples/


In [11]:
from IPython.display import display, HTML

display(HTML(response))

Name,Department,Experience
Ali,AI,2
Sara,Data Science,3
Hamza,Software Engineering,1


## Observation: 
#### The experiment demonstrates that an LLM can transform structured data from a Python dictionary into a different representation, such as an HTML table. The prompt specified the required title, column headers, rows, and output format, allowing the model to preserve the original information while changing its presentation. This shows how prompt engineering can be used for format conversion and structured output generation.

